In [1]:
# ¿Cómo transformamos mensajes financieros de texto bruto en una matriz documento–término que un clasificador pueda usar?

import pandas as pd

dataframe = pd.read_csv("../data/sentences.csv.gz", compression="gzip")
dataframe[["phrase", "target"]].head()

,phrase,target
0,"According to Gran , the company has no plans t...",neutral
1,"For the last quarter of 2010 , Componenta 's n...",positive
2,"In the third quarter of 2010 , net sales incre...",positive
3,Operating profit rose to EUR 13.1 mn from EUR ...,positive
4,"Operating profit totalled EUR 21.1 mn , up fro...",positive


In [2]:
# ¿Qué diferencias superficiales se pueden eliminar sin cambiar el contenido de cada mensaje?

dataframe["normalized_text"] = dataframe["phrase"].str.lower().str.split().str.join(" ")
dataframe[["phrase", "normalized_text"]].head()

,phrase,normalized_text
0,"According to Gran , the company has no plans t...","according to gran , the company has no plans t..."
1,"For the last quarter of 2010 , Componenta 's n...","for the last quarter of 2010 , componenta 's n..."
2,"In the third quarter of 2010 , net sales incre...","in the third quarter of 2010 , net sales incre..."
3,Operating profit rose to EUR 13.1 mn from EUR ...,operating profit rose to eur 13.1 mn from eur ...
4,"Operating profit totalled EUR 21.1 mn , up fro...","operating profit totalled eur 21.1 mn , up fro..."


In [3]:
# Se separa cada documento en tokens para que el texto se convierta en unidades procesables.

from nltk.tokenize import word_tokenize

dataframe["tokens"] = dataframe["normalized_text"].map(word_tokenize)
dataframe[["normalized_text", "tokens"]].head()

,normalized_text,tokens
0,"according to gran , the company has no plans t...","[according, to, gran, ,, the, company, has, no..."
1,"for the last quarter of 2010 , componenta 's n...","[for, the, last, quarter, of, 2010, ,, compone..."
2,"in the third quarter of 2010 , net sales incre...","[in, the, third, quarter, of, 2010, ,, net, sa..."
3,operating profit rose to eur 13.1 mn from eur ...,"[operating, profit, rose, to, eur, 13.1, mn, f..."
4,"operating profit totalled eur 21.1 mn , up fro...","[operating, profit, totalled, eur, 21.1, mn, ,..."


In [4]:
# Se conservan tokens alfabéticos y se hace visible la información que la regla descarta.

dataframe["tokens"] = dataframe["tokens"].map(
    lambda tokens: [token for token in tokens if token.isalpha()]
)
dataframe["processed_text"] = dataframe["tokens"].map(" ".join)
dataframe[["normalized_text", "processed_text"]].head()

,normalized_text,processed_text
0,"according to gran , the company has no plans t...",according to gran the company has no plans to ...
1,"for the last quarter of 2010 , componenta 's n...",for the last quarter of componenta net sales d...
2,"in the third quarter of 2010 , net sales incre...",in the third quarter of net sales increased by...
3,operating profit rose to eur 13.1 mn from eur ...,operating profit rose to eur mn from eur mn in...
4,"operating profit totalled eur 21.1 mn , up fro...",operating profit totalled eur mn up from eur m...


In [5]:
# ¿Cómo se transforma la lista de tokens en una matriz documento-término apta para clasificación?

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(min_df=2)
document_term_matrix = vectorizer.fit_transform(dataframe["processed_text"])
vocabulary = pd.DataFrame(
    {"token": vectorizer.get_feature_names_out(), "document_frequency": (document_term_matrix > 0).sum(axis=0).A1}
)
document_term_matrix.shape, vocabulary.head()

((2264, 2349),
    token  document_frequency
 0     ab                  13
 1    abb                   2
 2   able                   3
 3    abn                   2
 4  about                  66)

In [6]:
# Se guardan documentos procesados, vocabulario y matriz para que otro taller pueda entrenar un clasificador.

import json
from pathlib import Path
from scipy import sparse

submission_dir = Path("../submission")
dataframe.drop(columns="tokens").to_csv(submission_dir / "tokenized_sentences.csv", index=False)
vocabulary.to_csv(submission_dir / "vocabulary.csv", index=False)
sparse.save_npz(submission_dir / "document_term_matrix.npz", document_term_matrix)
with (submission_dir / "matrix_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "documents": document_term_matrix.shape[0],
            "terms": document_term_matrix.shape[1],
            "minimum_document_frequency": 2,
        },
        file,
        indent=2,
    )